# 00 — From PyTorch Operations to a Physics Residual

## Project theme

In this project, radiative transfer is a controlled setting for learning how physics-informed neural networks are constructed and diagnosed. The radiation physics is intentionally simple; the important work is translating a differential equation, a boundary condition, and observations into a differentiable optimization problem.

## Learning goals

By the end of this notebook, you should be able to:

- generate collocation points according to a specific sampling distribution;
- perform a complete training loop and update the optimizer;
- differentiate a vector of model outputs with respect to its input coordinates;
- construct the nondimensional radiative-transfer residual;
- explain why the differential-equation residual does not define the complete physical problem.

## 0. Setup

We will be using PyTorch for training our neural network, although we note that there are other alternatives for training machine learning models which we will not explore here. You should edit the code cell below to add all future PyTorch specific modifications/settings. For this and all future exercises, we highly encourate you to use the following sources of documentation (prioritized in listed order) to maximize retention and ability to expand your network with more complex capabilities in the future.

1) PyTorch documentation
2) StackOverflow questions
3) GenAI guidance (as a last resort)

While GenAI will likely have the correct answer most of the time, learning how to navigate the PyTorch documentation and reading the justifications people give for their answers on StackOverflow are vital to developing scientific machine learning intuition.

In [16]:
import torch

# We set the default tensor data type to float32 (not a double, or float64, as we don't need that level of precision).
torch.set_default_dtype(torch.float32)

# We set a fixed random number generation seed to ensure that results are replicated between student assignments.
# Comment out the torch.manual_seed() line to test how well your solution generalizes to varied random initializations.
torch.manual_seed(67)

## 1. The transport model

The steady-state (not varying with time), plane-parallel (a region well approximated by a stack of parallel planes, which holds when the thickness of the layer is small compared with the radius of curvature of the medium, $\Delta z / R \ll 1$) radiative transport equation is described by

$$
\frac{dI_\lambda}{dz} = -\rho(z) \kappa_\lambda(z) \left[I_\lambda(z) - S_\lambda\right]
$$

where the intensity $I$ at a given wavelength $\lambda$ varies as a function of $z$ as it propagates through a medium of bulk density $\rho(z)$ and macroscopic (bulk) opacity (at given wavelength $\lambda$) $\kappa_\lambda(z)$, under the influence of the medium's source intensity function $S_\lambda$. The source function $S_\lambda$ is the ratio of the emission to the absorption of photons in a given medium, and is thus problem-specific. For simplicity, in this example, we will assume the source function is the Planck function and that the medium's temperature is fixed as a function of $z$, therefore making $S_\lambda$ a constant value. This offers us a convenient analytic solution to which we can compare our PINN solution.

The plane-parallel formulation of the radiative transfer equation is a simplified version and is applicable in environments which exhibit changes only as a function of scale height $z$, such as atmospheres. Two further physical assumptions are folded into the form written above, and they are worth keeping distinct. The **steady-state** assumption removes the time dependence, so the intensity along a ray depends on position alone. **Local thermodynamic equilibrium** is a separate statement about the matter rather than about time: it is what allows the source function to be replaced by the Planck function evaluated at the local temperature, which is the substitution we rely on below.

This is the only form of the transport equation supplied to you. Every other form used in these notebooks is one you will derive from it.

### The analytic solution (provided)

Radiation enters the slab at $z=0$ with a specified intensity, which supplies the boundary condition

$$
I_\lambda(z=0) = I_{\lambda,0}.
$$

For constant $\rho$, $\kappa_\lambda$ and $S_\lambda$, the equation above together with this boundary condition has the closed-form solution

$$
I_\lambda(z) = S_\lambda + \left(I_{\lambda,0} - S_\lambda\right) e^{-\rho \kappa_\lambda z},
$$

where the exponent $\rho \kappa_\lambda z$ is the optical depth $\tau$ accumulated between the boundary and the height $z$. We provide this solution rather than asking you to derive it; you will use it throughout these notebooks as an independent reference against which the PINN solution is judged. Notice that the intensity relaxes from its boundary value $I_{\lambda,0}$ toward the source function $S_\lambda$ over roughly one optical depth, which is what makes $\tau$ the natural measure of distance for this problem.

In these modules, we will learn how to modify the form of the plane-parallel radiative transfer equation to match the assumptions and conventions most suitable for physics-informed neural network (PINN) applications.

### 2. Generating collocation points

Traditional neural networks create a map $\hat{y}$ between inputs $x$ and outputs $y$, with the goal of reducing the mismatch (or loss) between $\hat{y}$ and $y$ to as low a value as possible. PINNs peform a similar process, except they have the added benefit of ensuring that the learned behavior of $\hat{y}$ **must follow the physics constraints imposed by the differential equation** which defines the residual. Rather than providing our PINN with labeled inputs/outputs, we are going to learn the behavior across the entire domain by defining a set of points at which the differential equation is evaluated, known as collocation points, and minimizing the residual loss at those points.

To practice, create 100 collocation points on the `[0, 1]` domain and reshape them into a column tensor with shape `(100, 1)`. This domain is the nondimensional height coordinate $\hat{z}$ that you will construct in the next section, which is why it runs from 0 to 1 rather than over the physical range of $z$.

In [ ]:
# TODO: create a column tensor with shape (100, 1).
# Later sections differentiate quantities with respect to these coordinates.
x = ...

<details>
<summary><strong>Conceptual hint</strong></summary>

A neural network treats the first dimension as the collection of samples and the second dimension as the number of input features/dimensions.

In section 3 you will differentiate a quantity with respect to these coordinates. Autograd can only trace that derivative if the coordinate tensor is marked as something it should track from the moment the tensor is created; a plain tensor of numbers carries no history and cannot be differentiated against.

</details>

<details>
<summary><strong>PyTorch hint</strong></summary>

Consider looking up the `linspace` and `reshape` functions on the PyTorch documentation pages.

To make the tensor differentiable with respect to its own entries, look up `requires_grad_` (note the trailing underscore, which marks an in-place operation in PyTorch) and the `requires_grad` attribute it sets.

</details>

## 3. Defining your first neural network (not a PINN yet!)

PyTorch makes defining a neural network, which stores trainable weights and biases, trivial using objects defined in the `torch.nn` library. The components of a neural network we will be exploring are:

1. the model (including layers and activation functions);
2. the loss function;
3. the optimizer (including the learning rate);
4. (optional) a learning rate scheduler.

Below, define a simple model using linear layers with your choice of width and your choice of activation function. Then, define an optimizer from `torch.optim` and set an initial learning rate for it. Write a simple training loop (including loss backpropagation, search PyTorch documentation for examples) for at most a few hundred epochs which learns a linear map $x \to x$. Use the mean squared error loss from `torch.nn` and print the model's loss every 100 epochs.

**Questions:**
1) What is a criterion that must be satisifed for PINNs when choosing the loss function?
2) Comment on your network's performance as you change i) the number and width of layers, ii) the initial learning rate, and iii) the power $p$ of $y = x^p$. 

In [14]:
# TODO: define a simple linear model

# TODO: define mean squared error loss

# TODO: define an optimizer

# TODO: define the training loop including loss backpropagation, gradient clearing, and updating optimizer parameters

# TODO: learn the x->x linear map and then modify as y=x^p for question 2


<details>
<summary><strong>Conceptual hint</strong></summary>

The scalar objective is the mean of the squared difference between prediction and target. Gradients must be cleared before they are accumulated again.

</details>

<details>
<summary><strong>PyTorch hint</strong></summary>

Some relevant functions are `optimizer.zero_grad()`, `loss(x, y)`, `loss.backward()`, and `optimizer.step()`.

</details>

Now we want to replace the components of our standard neural network with those of a PINN. Namely, our first step will be nondimensionalizing our equation, as is standard procedure for any numerical solution of a differential equation ([see here for more on this](https://people.math.wisc.edu/~angenent/519.2016s/notes/non-dimensionalization.html)).

To start, we will need to define a more specific problem to have a better understanding of our parameter domains. Let's consider, for example, the Earth's atmosphere, specifically the troposphere. With some simplifications, we can roughly characterize the troposphere as having the following characteristics:

- $z \in [0.1, 2 \times 10^6]$ cm
- $\kappa \in [10^{-3}, 10^{-2}]$ cm$^2$/g
- $\rho \in [4 \times 10^{-4}, 10^{-3}]$ g/cm$^3$

Depending on which combination of $\kappa$ and $\rho$ values we assume, our optical depth $\tau = \rho \kappa z$ ranges from $0.8 < \tau < 20$. This range represents an ideal case to study the evolution of the radiation through the atmosphere: from the analytic solution given in §1, the intensity relaxes toward $S_\lambda$ over one optical depth, so depending on your choice of $\rho$ and $\kappa$ the solution will decay either as a slightly damped exponential or a rapidly decaying exponential (ideally somewhere around $\tau = 1$ to start, and closer to $\tau$ of a few for comparison experiments). All that is remaining is to set our source function $S_\lambda$, which we have previously determined to be the Planck function, evaluated at T = 300K for the Earth's atmosphere.

To finish the setup, now determine the following:

- What is an appropriate wavelength $\lambda$ at which we should examine radiation in the Earth's atmosphere? Hint: Wien's law.
- Determine normalization constants for the height $z$, the intensity ($I_\lambda, S_\lambda$), the density $\rho$ and the opacity $\kappa$. The normalization for $I_\lambda$ and $S_\lambda$ should be the same. Hint: Is $S_\lambda$ varying?
- Choose those constants so that each nondimensional quantity — $\hat{z}$, $\hat{u}$, $\hat{\rho}$, $\hat{\kappa}$ — lands inside `[0, 1]`. Hint: for the height in particular, the nondimensional coordinate $\hat{z} = z / z_0$ must be normalized to the range `[0, 1]`, which fixes $z_0$ from the physical domain quoted above and matches the collocation points you built in the previous section. The one quantity deliberately allowed outside `[0, 1]` is the optical depth $\tau$: it is a ratio of two physical scales rather than a rescaled variable, and its being larger than one is exactly what makes the problem interesting.
- Because the temperature is fixed and the source function is the Planck function, $S_\lambda$ does not vary with $z$. Normalizing both $I_\lambda$ and $S_\lambda$ by that same constant therefore sets the nondimensional source to `SOURCE = 1`, which is supplied for you below.
- Set `INFLOW`, the physical intensity $I_{\lambda,0}$ entering the slab at $z = 0$, and `u_0`, its nondimensional counterpart under the normalization you just chose. `u_0` is the boundary value that the nondimensional solution must satisfy.
- Set `OPACITY`, the dimensionless group that multiplies $(u - $ `SOURCE` $)$ once the equation is written in nondimensional form. Confirm that its value is consistent with the range of $\tau$ quoted above.

Below, write code to set up the nondimensional version of the problem as discussed in this cell to finalize PINN setup.

In [ ]:
# TODO: set up the Planck function for an appropriate wavelength lambda at T=300K.

def blackbody_lambda_cgs(lam_cm, temp_k):
    """
    Computes the Planck blackbody spectral radiance in CGS units.
    
    Parameters:
        lam_cm (Tensor): Wavelength in centimeters (cm)
        temp_k (Tensor or float): Temperature in Kelvin (K)
        
    Returns:
        Tensor: Spectral radiance in erg / (cm^2 * s * cm)
    """
    # CGS Constants
    h = 6.62607015e-27  # Planck's constant (erg s)
    c = 2.99792458e10   # Speed of light (cm/s)
    k = 1.380649e-16    # Boltzmann's constant (erg/K)
    
    # Exponent term: hc / (lambda * k * T)
    # Clamp/prevent overflow for very small wavelengths or high values
    exponent = (h * c) / (lam_cm * k * temp_k)
    
    # Prefactor: 2 * h * c^2 / lambda^5
    prefactor = (2.0 * h * (c ** 2)) / (lam_cm ** 5)
    
    # Planck's law per unit wavelength
    intensity = prefactor / (torch.exp(exponent) - 1.0)
    return intensity

# TODO: define normalization constants z_0, k_0, rho_0, and S_0

# The intensity and the source function share a normalization constant, and the
# source function is constant, so the nondimensional source is fixed:
SOURCE = 1.0

# TODO: define the physical inflow intensity I_lambda(z=0) in CGS units,
#       and its nondimensional counterpart u_0.
INFLOW = ...
u_0 = ...

# TODO: define OPACITY, the dimensionless group appearing in your
#       nondimensional form of the transport equation.
OPACITY = ...

## 4. Automatic differentiation with respect to coordinates

PINNs require derivatives of the predicted solution with respect to its independent variables. These are **different** from the neural network parameter gradients (weight gradients). Here we will use the analytic solution (see §1) with respect to some inputs `z`, in preparation for writing the non-dimensional version of our equation for use in our PINN.

The key component of auto-differentiation that is crucial to both i) tracking neural network weight gradients and ii) tracking the functional gradients with respect to the independent variables is using the directed acyclic graph feature of autograd. It tracks all the operations which occur during a forward pass, enabling automatic differentiation via the chain rule during a backward pass.

To start, we are going to use the simple example of $y = x^2$ for the autograd example. Populate the `coordinate_derivative` function with the `autograd` automatic differentiation, then test its validity against the analytic form of the derivative for the same inputs $x$.

In [17]:
# TODO: populate the coordinate_derivative function with an autograd function for y=x^2

def coordinate_derivative(values: torch.Tensor, coordinates: torch.Tensor) -> torch.Tensor:
    """Return d(values)/d(coordinates) with the same shape as coordinates."""
    # TODO: call torch.autograd.grad.
    # The returned derivative must remain connected to a graph because a PINN
    # later differentiates a loss containing this derivative with respect to
    # the neural-network parameters.
    return ...

#TODO: compare to analytic form of y=x^2 derivative

<details>
<summary><strong>PyTorch hint</strong></summary>

Use `torch.autograd.grad(outputs=values, inputs=coordinates, grad_outputs=torch.ones_like(values), create_graph=True)[0]`.

</details>

### Deriving the nondimensional equation

Now, armed with all the necessary tools, write the nondimensional form of the radiative transfer equation yourself. Only the physical form is supplied to you, in §1; the nondimensional form is the thing you are deriving. Substitute

$$
z = z_0\hat{z}, \qquad I_\lambda = S_0 u, \qquad S_\lambda = S_0\,\texttt{SOURCE}, \qquad \rho = \rho_0\hat{\rho}, \qquad \kappa_\lambda = \kappa_0\hat{\kappa}
$$

into the transport equation, apply the chain rule to $dI_\lambda/dz$, and collect every leftover constant into the single dimensionless group you called `OPACITY`. Transform the result into a residual by moving all terms to one side so that the equation reads $r(\hat{z}) = 0$, then implement that residual in `transport_residual` below.

The nondimensional analytic solution is supplied in the next cell as `exact_solution_torch` so that you can check your derivation. Evaluating your residual on it should give a value close to zero at every coordinate. If it does not, then your residual and the supplied solution disagree, and the sign or the grouping of constants in your derivation is where to look first.

In [ ]:
def exact_solution_torch(
    x: torch.Tensor,
    opacity: float | torch.Tensor,
    source: float | torch.Tensor,
    inflow: float | torch.Tensor) -> torch.Tensor:
    """Nondimensional analytic solution, provided. Preserves x's dtype/device.

    This is the nondimensional counterpart of the closed-form solution quoted in
    §1, written in terms of the constants you defined above. Use it to check the
    residual you derive; do not use it inside transport_residual itself.
    """
    a = torch.as_tensor(opacity, dtype=x.dtype, device=x.device)
    q = torch.as_tensor(source, dtype=x.dtype, device=x.device)
    u_in = torch.as_tensor(inflow, dtype=x.dtype, device=x.device)
    return q + (u_in - q) * torch.exp(-a * x)

def coordinate_derivative(values: torch.Tensor, coordinates: torch.Tensor) -> torch.Tensor:
    """Return d(values)/d(coordinates) with the same shape as coordinates."""
    # TODO: call torch.autograd.grad.
    # The returned derivative must remain connected to a graph because a PINN
    # later differentiates a loss containing this derivative with respect to
    # the neural-network parameters.
    return ...

def transport_residual(
    coordinates: torch.Tensor,
    values: torch.Tensor,
    opacity: float,
    source: float) -> torch.Tensor:
    # TODO: use coordinate_derivative and assemble the residual you derived
    # from the physical transport equation in section 1.
    return ...

# Evaluate the supplied analytic solution at your collocation points, then check
# that your residual very nearly vanishes there.
u = exact_solution_torch(x, OPACITY, SOURCE, u_0)

residual_exact = transport_residual(
    x,
    u,
    opacity=OPACITY,
    source=SOURCE,
)

print("maximum |residual|:", float(residual_exact.detach().abs().max()))

<details>
<summary><strong>Conceptual hint</strong></summary>

`values` and `coordinates` contain many entries, but PyTorch differentiates a scalar contraction, so it needs one weight per output entry; weighting every entry by one recovers the derivative of each output with respect to its own input. Ensure that you retain the graph for later parameter differentiation. The residual is a pointwise quantity. Do not average it yet; preserve one residual value per coordinate.

</details>

<details>
<summary><strong>PyTorch hint</strong></summary>

Use `torch.autograd.grad(outputs=values, inputs=coordinates, grad_outputs=torch.ones_like(values), create_graph=True)[0]`.

For the residual, compute `du_dx = coordinate_derivative(values, coordinates)` and then combine it with an `opacity` term and a `source` term arranged as your derivation dictates. The supplied analytic solution decides whether you got the arrangement right.

</details>

## 5. The residual is not the complete problem

For any constant `C`,

$$
u_C(x)=q+Ce^{-ax}
$$

satisfies the differential equation. The boundary condition selects exactly one member of this family.

Complete the loop below and compare both residual and boundary error.

In [ ]:
constants = [-2.0, 0.0, u_0 - SOURCE, 5.0]
rows = []

for constant in constants:
    x_family = torch.linspace(0.0, 1.0, 201).reshape(-1, 1).requires_grad_(True)

    # TODO: construct u_C(x) = source + constant * exp(-opacity*x).
    u_family = None

    # TODO: evaluate its pointwise residual.
    r_family = None

    rms_residual = None      # TODO: square, average, and take the square root.
    boundary_error = None    # TODO: compute |u_C(0) - u_0|.

    rows.append((constant, rms_residual, boundary_error))

print(f"{'C':>8} {'RMS residual':>16} {'boundary error':>18}")
for row in rows:
    print(f"{row[0]:8.3f} {row[1]:16.3e} {row[2]:18.3e}")

<details>
<summary><strong>Conceptual hint</strong></summary>

All four functions should satisfy the differential equation. Only `C = u_0 - source` should satisfy the stated boundary value.

</details>

<details>
<summary><strong>PyTorch hint</strong></summary>

Use `torch.exp(-OPACITY * x_family)`, `torch.sqrt(torch.mean(r_family.square()))`, and the first entry of `u_family`.

</details>

### Interpretation checkpoint

Write brief answers before continuing:

1. Why can several different functions have nearly zero residual?
2. What mathematical role does the boundary condition play?
3. Why would reporting only the physics residual be insufficient evidence that a PINN solved the problem?
4. In the derivative function, why must the coordinate tensor require gradients?
5. Why is `create_graph=True` needed during PINN training?

## 6. Reusable takeaway

A PINN does not receive a complete physical problem merely because a differential equation appears in its loss. A well-posed model must represent all required information: differential equations, boundary or initial conditions, parameter constraints, and—when applicable—observations.